# Notebook 1 — Getting Data
**BSE Stock Market Volatility Forecasting**

**Goals:**
1. Install and import required libraries
2. Fetch BSE / NSE stock data from Yahoo Finance using `yfinance`
3. Explore the raw data (shape, dtypes, nulls)
4. Clean the data (column naming, null handling, type casting)
5. Compute daily percentage returns
6. Build a reusable `get_stock_data()` function
7. Visualise price and return series

## 1. Install dependencies

Run this cell once (skip in Replit where `requirements.txt` is already installed).

In [ ]:
# Uncomment to install in Google Colab
# !pip install yfinance pandas numpy matplotlib plotly

## 2. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import plotly.express as px
import yfinance as yf

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('All libraries imported successfully!')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'yfinance: {yf.__version__}')

## 3. Key BSE / NSE Tickers

Yahoo Finance uses `.NS` for NSE-listed stocks and `.BO` for BSE-listed stocks.
The SENSEX index symbol is `^BSESN`.

In [ ]:
TICKERS = {
    '^BSESN'       : 'BSE SENSEX',
    '^NSEI'        : 'NSE NIFTY 50',
    'RELIANCE.NS'  : 'Reliance Industries',
    'TCS.NS'       : 'Tata Consultancy Services',
    'INFY.NS'      : 'Infosys',
    'HDFCBANK.NS'  : 'HDFC Bank',
    'WIPRO.NS'     : 'Wipro',
    'ITC.NS'       : 'ITC Ltd',
    'SBIN.NS'      : 'State Bank of India',
    'TATAMOTORS.NS': 'Tata Motors',
}

for symbol, name in TICKERS.items():
    print(f'  {symbol:<18}  {name}')

## 4. Download Raw Data

In [ ]:
TICKER = '^BSESN'
START  = '2015-01-01'
END    = '2024-12-31'

raw = yf.download(TICKER, start=START, end=END, auto_adjust=True, progress=False)

print(f'Downloaded {len(raw):,} rows for {TICKER}')
print(f'Date range : {raw.index.min().date()}  →  {raw.index.max().date()}')
raw.head()

## 5. Explore Raw Structure

In [ ]:
print('=== Shape ===')
print(raw.shape)

print('\n=== dtypes ===')
print(raw.dtypes)

print('\n=== Null counts ===')
print(raw.isnull().sum())

print('\n=== Descriptive statistics ===')
raw.describe()

## 6. Clean the Data

- Flatten MultiIndex columns (yfinance sometimes returns them)
- Keep only OHLCV columns
- Name the index `Date`
- Drop any remaining null rows

In [ ]:
def clean_ohlcv(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Flatten columns, keep OHLCV, drop nulls."""
    df = raw_df.copy()

    # Flatten MultiIndex if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index.name = 'Date'
    df.dropna(inplace=True)
    return df


df = clean_ohlcv(raw)
print(f'After cleaning: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Remaining nulls: {df.isnull().sum().sum()}')
df.head()

## 7. Compute Daily Returns

In [ ]:
df['returns'] = df['Close'].pct_change() * 100   # percentage returns
df.dropna(inplace=True)

print('Returns column added.')
print(f'Mean return : {df["returns"].mean():.4f}%')
print(f'Std return  : {df["returns"].std():.4f}%')
print(f'Min return  : {df["returns"].min():.4f}%')
print(f'Max return  : {df["returns"].max():.4f}%')
df[['Close', 'returns']].tail()

## 8. The `get_stock_data()` Function

This is the function we will move into `src/data.py`.

In [ ]:
def get_stock_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Fetch and clean BSE/NSE stock data from Yahoo Finance.

    Parameters
    ----------
    ticker : str
        Yahoo Finance symbol, e.g. 'RELIANCE.NS', '^BSESN'.
    start : str
        Start date YYYY-MM-DD.
    end : str
        End date YYYY-MM-DD.

    Returns
    -------
    pd.DataFrame
        Columns: Open, High, Low, Close, Volume, returns.
        Index: DatetimeIndex named 'Date'.
    """
    raw = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)

    if raw.empty:
        raise ValueError(f"No data returned for ticker: {ticker}")

    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
    df.index.name = 'Date'
    df.dropna(inplace=True)
    df['returns'] = df['Close'].pct_change() * 100
    df.dropna(inplace=True)
    return df


# --- Test the function ---
sensex = get_stock_data('^BSESN', '2020-01-01', '2024-12-31')
print(f'Fetched {len(sensex):,} rows for SENSEX')
print(sensex.dtypes)
sensex.head()

## 9. Visualise: Closing Price

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sensex.index,
    y=sensex['Close'],
    mode='lines',
    name='SENSEX Close',
    line=dict(color='#1f77b4', width=1.5)
))
fig.update_layout(
    title='BSE SENSEX — Closing Price (2020–2024)',
    xaxis_title='Date',
    yaxis_title='Price (INR)',
    template='plotly_white',
    height=450
)
fig.show()

## 10. Visualise: Daily Returns & Rolling Volatility

In [ ]:
sensex['rolling_vol_30d'] = sensex['returns'].rolling(30).std() * np.sqrt(252)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(sensex.index, sensex['returns'], color='steelblue', linewidth=0.8, alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.6, linestyle='--')
axes[0].set_title('SENSEX — Daily Percentage Returns', fontsize=13)
axes[0].set_ylabel('Return (%)')

axes[1].plot(sensex.index, sensex['rolling_vol_30d'], color='firebrick', linewidth=1.2)
axes[1].set_title('30-Day Rolling Volatility (Annualised)', fontsize=13)
axes[1].set_ylabel('Volatility (%)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.show()

print('Notice the volatility spikes around March 2020 (COVID crash) and 2022.')

## 11. Compare Multiple BSE Stocks

In [ ]:
compare_tickers = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS']
close_prices = {}

for t in compare_tickers:
    data = get_stock_data(t, '2020-01-01', '2024-12-31')
    close_prices[t] = data['Close']

price_df = pd.DataFrame(close_prices)

# Normalise to 100 at start for comparison
norm = price_df / price_df.iloc[0] * 100

fig = px.line(
    norm,
    title='Normalised Price Performance — Top NSE Stocks (Base=100, Jan 2020)',
    template='plotly_white',
    height=450,
    labels={'value': 'Normalised Price', 'index': 'Date', 'variable': 'Ticker'}
)
fig.show()

## 12. Return Distribution

In [ ]:
from scipy import stats

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(sensex['returns'], bins=100, density=True, color='steelblue', alpha=0.6, label='Returns')

# Overlay normal distribution for comparison
mu, sigma = sensex['returns'].mean(), sensex['returns'].std()
x = np.linspace(sensex['returns'].min(), sensex['returns'].max(), 200)
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal distribution')

ax.set_title('SENSEX — Return Distribution vs. Normal', fontsize=13)
ax.set_xlabel('Daily Return (%)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Skewness : {sensex["returns"].skew():.4f}   (negative = left-skewed)')
print(f'Kurtosis : {sensex["returns"].kurtosis():.4f}  (> 0 = fat tails — expected for stocks)')

## Summary

- We can fetch clean OHLCV data for any BSE/NSE stock using `get_stock_data(ticker, start, end)`.
- Returns show **fat tails** and **volatility clustering** — hallmarks that justify GARCH modelling.
- The `get_stock_data()` function is production-ready and lives in `src/data.py`.

**Next:** Notebook 2 — building the SQLite data model and `data.py` with TDD.